# Data Cleaning And Feature Selection: General Intake

This notebook is the shared starting point for the cleaning and feature-selection workflow.

Current scope:
- load the five contest tables from `data/`
- keep a raw view for special missing-token checks
- build a normalized view with the same rules as `src/python/data_intake.py`
- summarize missing values at dataset and column level
- expose column names and dtypes for the later linking and feature-selection decisions

Not in scope yet:
- imputation or dropping rules
- dataset joins
- feature shortlist decisions


In [133]:
from importlib import util
from pathlib import Path
import sys
import warnings

import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_INTAKE_PATH = PROJECT_ROOT / "src" / "python" / "data_intake.py"
spec = util.spec_from_file_location("project_data_intake", DATA_INTAKE_PATH)
data_intake = util.module_from_spec(spec)
assert spec is not None and spec.loader is not None
sys.modules[spec.name] = data_intake
spec.loader.exec_module(data_intake)

DATASET_FILES = data_intake.DATASET_FILES
NA_TOKENS = data_intake.NA_TOKENS
normalize_dataframe = data_intake.normalize_dataframe

DATA_DIR = PROJECT_ROOT / "data"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Datasets: {DATASET_FILES}")


Project root: /Users/jan/Documents/competitions/hackatons/WEC_26
Data directory: /Users/jan/Documents/competitions/hackatons/WEC_26/data
Datasets: ['players_quarters_final.csv', 'player_appearance_pass.csv', 'player_appearance_behaviour_under_pressure.csv', 'player_appearance_run.csv', 'player_appearance_shot_limited.csv']


In [134]:
TRACKED_MISSING_TOKENS = ["", "NULL", "null", "None", "none", "N/A", "n/a", "NA", "na", "nan"]


def load_raw_table(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, keep_default_na=False)


def load_normalized_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, na_values=NA_TOKENS)
    return normalize_dataframe(df)


raw_tables = {name: load_raw_table(DATA_DIR / name) for name in DATASET_FILES}
tables = {name: load_normalized_table(DATA_DIR / name) for name in DATASET_FILES}


def dataset_inventory(name: str, df: pd.DataFrame) -> dict:
    missing_cells = int(df.isna().sum().sum())
    return {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": missing_cells,
        "missing_pct": round(100 * missing_cells / max(df.size, 1), 2),
        "duplicate_rows": int(df.duplicated().sum()),
    }


inventory = pd.DataFrame(
    dataset_inventory(name, df) for name, df in tables.items()
).sort_values("dataset").reset_index(drop=True)

display(Markdown("## Dataset Inventory"))
display(inventory)


## Dataset Inventory

,dataset,rows,columns,missing_cells,missing_pct,duplicate_rows
0,player_appearance_behaviour_under_pressure.csv,12185,10,6456,5.30,0
1,player_appearance_pass.csv,29795,7,98,0.05,0
2,player_appearance_run.csv,35133,10,0,0.00,0
3,player_appearance_shot_limited.csv,780,12,1344,14.36,0
4,players_quarters_final.csv,3486,33,0,0.00,0


In [135]:
def raw_missing_token_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in df.columns:
        values = df[column].astype("string").str.strip()
        for token in TRACKED_MISSING_TOKENS:
            count = int((values == token).sum())
            if count > 0:
                rows.append(
                    {
                        "column": column,
                        "token": "<EMPTY>" if token == "" else token,
                        "count": count,
                    }
                )

    if not rows:
        return pd.DataFrame(columns=["column", "token", "count"])

    return pd.DataFrame(rows).sort_values(
        ["count", "column", "token"],
        ascending=[False, True, True],
    ).reset_index(drop=True)


display(Markdown("## Raw Missing-Token Scan"))
for name, df in raw_tables.items():
    summary = raw_missing_token_summary(df)
    display(Markdown(f"### `{name}`"))
    if summary.empty:
        print("No tracked raw missing tokens found.")
    else:
        display(summary)


## Raw Missing-Token Scan

### `players_quarters_final.csv`

No tracked raw missing tokens found.


### `player_appearance_pass.csv`

,column,token,count
0,stage,<EMPTY>,57
1,addressee_player_appearance_id,NULL,41


...

,column,token,count
0,pass_angle,NULL,5793
1,addressee_player_appearance_id,NULL,649
2,stage,<EMPTY>,14


### `player_appearance_run.csv`

No tracked raw missing tokens found.


### `player_appearance_shot_limited.csv`

,column,token,count
0,own_goal_player_appearance_id,NULL,779
1,block_player_appearance_id,NULL,565


In [136]:
def column_missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in df.columns:
        missing_count = int(df[column].isna().sum())
        rows.append(
            {
                "column": column,
                "dtype": str(df[column].dtype),
                "non_null": int(df[column].notna().sum()),
                "missing_count": missing_count,
                "missing_pct": round(100 * missing_count / max(len(df), 1), 2),
                "n_unique": int(df[column].nunique(dropna=True)),
            }
        )

    return pd.DataFrame(rows).sort_values(
        ["missing_count", "column"],
        ascending=[False, True],
    ).reset_index(drop=True)


dataset_missing_overview = pd.DataFrame(
    {
        "dataset": name,
        "columns_with_missing": int(df.isna().any().sum()),
        "rows_with_missing": int(df.isna().any(axis=1).sum()),
        "total_missing_cells": int(df.isna().sum().sum()),
    }
    for name, df in tables.items()
).sort_values("dataset").reset_index(drop=True)

display(Markdown("## Normalized Missingness By Dataset"))
display(dataset_missing_overview)

display(Markdown("## Column-Level Schema And Missingness"))
for name, df in tables.items():
    display(Markdown(f"### `{name}`"))
    print(f"shape = {df.shape}")
    display(column_missing_summary(df))


## Normalized Missingness By Dataset

,dataset,columns_with_missing,rows_with_missing,total_missing_cells
0,player_appearance_behaviour_under_pressure.csv,3,5793,6456
1,player_appearance_pass.csv,2,98,98
2,player_appearance_run.csv,0,0,0
3,player_appearance_shot_limited.csv,2,780,1344
4,players_quarters_final.csv,0,0,0


## Column-Level Schema And Missingness

### `players_quarters_final.csv`

shape = (3486, 33)


...

shape = (35133, 10)


,column,dtype,non_null,missing_count,missing_pct,n_unique
0,distance,float64,35133,0,0.0,33613
1,id,int64,35133,0,0.0,35133
2,max_speed,float64,35133,0,0.0,489
3,min_speed,float64,35133,0,0.0,297
4,minute,int64,35133,0,0.0,57
5,period,category,35133,0,0.0,4
6,player_appearance_id,int64,35133,0,0.0,968
7,possession,int64,35133,0,0.0,62
8,run_type,category,35133,0,0.0,2
9,stage,category,35133,0,0.0,3


### `player_appearance_shot_limited.csv`

shape = (780, 12)


,column,dtype,non_null,missing_count,missing_pct,n_unique
0,own_goal_player_appearance_id,float64,1,779,99.87,1
1,block_player_appearance_id,float64,215,565,72.44,172
2,body_part,category,780,0,0.00,4
3,id,int64,780,0,0.00,780
4,minute,int64,780,0,0.00,53
5,period,category,780,0,0.00,4
6,play_pattern,category,780,0,0.00,7
7,player_appearance_id,int64,780,0,0.00,453
8,possession,int64,780,0,0.00,62
9,stage,category,780,0,0.00,3


In [137]:
display(Markdown("## `minute_in` And `minute_out` Grouped By Checkpoint"))

df = tables["players_quarters_final.csv"].copy()

checkpoint_minute_summary = (
    df.groupby("checkpoint", observed=False)
    .agg(
        rows=("player_appearance_id", "size"),
        minute_in_min=("minute_in", "min"),
        minute_in_median=("minute_in", "median"),
        minute_in_mean=("minute_in", "mean"),
        minute_in_max=("minute_in", "max"),
        minute_out_min=("minute_out", "min"),
        minute_out_median=("minute_out", "median"),
        minute_out_mean=("minute_out", "mean"),
        minute_out_max=("minute_out", "max"),
    )
    .reset_index()
)

for column in [
    "minute_in_median",
    "minute_in_mean",
    "minute_out_median",
    "minute_out_mean",
]:
    checkpoint_minute_summary[column] = checkpoint_minute_summary[column].round(2)

display(checkpoint_minute_summary)


## `minute_in` And `minute_out` Grouped By Checkpoint

,checkpoint,rows,minute_in_min,minute_in_median,minute_in_mean,minute_in_max,minute_out_min,minute_out_median,minute_out_mean,minute_out_max
0,ET1_15,42,1,22.5,41.88,102,106,120.0,119.52,120
1,H1_15,681,1,1.0,1.01,11,21,90.0,82.47,120
2,H1_30,680,1,1.0,1.01,11,44,90.0,82.56,120
3,H1_45,680,1,1.0,1.08,44,46,90.0,82.68,120
4,H2_15,680,1,1.0,5.04,59,60,90.0,85.92,120
5,H2_30,680,1,1.0,16.49,74,75,90.0,90.16,120
6,H2_45,43,1,1.0,18.79,87,90,120.0,113.30,120


In [138]:
display(Markdown("## Checkpoint-Window Features For `minute_in` And `minute_out`"))

checkpoint_windows = {
    "H1_15": (1, 15),
    "H1_30": (16, 30),
    "H1_45": (31, 45),
    "H2_15": (46, 60),
    "H2_30": (61, 75),
    "H2_45": (76, 90),
    "ET1_15": (91, 105),
    "ET2_15": (106, 120),
}

df = tables["players_quarters_final.csv"].copy()
engineered_minute_columns = []

for base_column in ["minute_in", "minute_out"]:
    for checkpoint_label, (start_minute, end_minute) in checkpoint_windows.items():
        feature_name = f"{base_column}_{checkpoint_label}"
        engineered_minute_columns.append(feature_name)
        df[feature_name] = (
            df[base_column]
            .between(start_minute, end_minute)
            .astype("int64")
        )

tables["players_quarters_final.csv"] = df

feature_summary = pd.DataFrame(
    {
        "feature": feature_name,
        "active_rows": int(df[feature_name].sum()),
        "zero_rows": int((df[feature_name] == 0).sum()),
        "unique_values": ", ".join(str(value) for value in sorted(df[feature_name].unique())),
    }
    for feature_name in engineered_minute_columns
).sort_values("feature").reset_index(drop=True)

preview_columns = [
    "player_appearance_id",
    "checkpoint",
    "minute_in",
    "minute_out",
    *engineered_minute_columns[:6],
]

outside_window_summary = pd.DataFrame(
    {
        "base_column": base_column,
        "rows_outside_defined_checkpoint_windows": int(
            (~pd.concat(
                [
                    df[base_column].between(start_minute, end_minute)
                    for start_minute, end_minute in checkpoint_windows.values()
                ],
                axis=1,
            ).any(axis=1)).sum()
        ),
    }
    for base_column in ["minute_in", "minute_out"]
)

display(Markdown("### Engineered Feature Summary"))
display(feature_summary)

display(Markdown("### Preview"))
display(df[preview_columns].head(10))

display(Markdown("### Values Outside Defined Checkpoint Windows"))
display(outside_window_summary)


## Checkpoint-Window Features For `minute_in` And `minute_out`

### Engineered Feature Summary

,feature,active_rows,zero_rows,unique_values
0,minute_in_ET1_15,10,3476,"0, 1"
1,minute_in_ET2_15,0,3486,0
2,minute_in_H1_15,3223,263,"0, 1"
3,minute_in_H1_30,0,3486,0
4,minute_in_H1_45,5,3481,"0, 1"
5,minute_in_H2_15,118,3368,"0, 1"
6,minute_in_H2_30,122,3364,"0, 1"
7,minute_in_H2_45,8,3478,"0, 1"
8,minute_out_ET1_15,60,3426,"0, 1"
9,minute_out_ET2_15,188,3298,"0, 1"


### Preview

,player_appearance_id,checkpoint,minute_in,minute_out,minute_in_H1_15,minute_in_H1_30,minute_in_H1_45,minute_in_H2_15,minute_in_H2_30,minute_in_H2_45
0,39421,H1_15,1,90,1,0,0,0,0,0
1,39422,H1_15,1,90,1,0,0,0,0,0
2,39423,H1_15,1,90,1,0,0,0,0,0
3,39424,H1_15,1,90,1,0,0,0,0,0
4,39425,H1_15,1,75,1,0,0,0,0,0
5,39426,H1_15,1,87,1,0,0,0,0,0
6,39429,H1_15,1,90,1,0,0,0,0,0
7,39431,H1_15,1,80,1,0,0,0,0,0
8,39433,H1_15,1,87,1,0,0,0,0,0
9,39435,H1_15,1,90,1,0,0,0,0,0


### Values Outside Defined Checkpoint Windows

,base_column,rows_outside_defined_checkpoint_windows
0,minute_in,0
1,minute_out,0


In [139]:
def build_column_presence_map(table_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    all_columns = sorted({column for df in table_dict.values() for column in df.columns})
    rows = []
    for column in all_columns:
        row = {"column": column}
        for name, df in table_dict.items():
            row[name.replace(".csv", "")] = column in df.columns
        row["dataset_count"] = sum(
            int(row[name.replace(".csv", "")]) for name in table_dict
        )
        rows.append(row)

    return pd.DataFrame(rows).sort_values(
        ["dataset_count", "column"],
        ascending=[False, True],
    ).reset_index(drop=True)


display(Markdown("## Shared Columns For Later Linking"))
display(build_column_presence_map(tables))


## Shared Columns For Later Linking

,column,players_quarters_final,player_appearance_pass,player_appearance_behaviour_under_pressure,player_appearance_run,player_appearance_shot_limited,dataset_count
0,player_appearance_id,True,True,True,True,True,5
1,id,False,True,True,True,True,4
2,minute,False,True,True,True,True,4
3,period,False,True,True,True,True,4
4,stage,False,True,True,True,True,4
5,accurate,False,True,True,False,False,2
6,addressee_player_appearance_id,False,True,True,False,False,2
7,possession,False,False,False,True,True,2
8,block_player_appearance_id,False,False,False,False,True,1
9,body_part,False,False,False,False,True,1


## Step 1: Replace Legacy `minute_in` / `minute_out`

In this step `df` is the imported target dataset and `clean_df` is the modeling copy. We compress the checkpoint-window minute columns into dense replacement features and build `clean_df` without the legacy raw minute columns.


In [140]:
minute_window_order = list(checkpoint_windows.keys())
minute_window_to_idx = {window: idx for idx, window in enumerate(minute_window_order, start=1)}


def compress_checkpoint_features(df: pd.DataFrame, base_column: str) -> tuple[pd.Series, pd.Series]:
    feature_columns = [f"{base_column}_{window}" for window in minute_window_order]
    hit_count = df[feature_columns].sum(axis=1)

    compressed = pd.Series(pd.NA, index=df.index, dtype="string")
    for window in minute_window_order:
        feature_column = f"{base_column}_{window}"
        compressed = compressed.mask(df[feature_column] == 1, window)

    diagnostics = pd.Series(
        {
            "base_column": base_column,
            "rows_with_zero_active_windows": int((hit_count == 0).sum()),
            "rows_with_multiple_active_windows": int((hit_count > 1).sum()),
        }
    )
    return compressed, diagnostics


minute_in_window, minute_in_diag = compress_checkpoint_features(
    df,
    "minute_in",
)
minute_out_window, minute_out_diag = compress_checkpoint_features(
    df,
    "minute_out",
)

compression_diagnostics = pd.DataFrame([minute_in_diag, minute_out_diag])
display(Markdown("### Compression Diagnostics"))
display(compression_diagnostics)


### Compression Diagnostics

,base_column,rows_with_zero_active_windows,rows_with_multiple_active_windows
0,minute_in,0,0
1,minute_out,0,0


In [141]:
clean_df = df.copy()
clean_df["minutes_in_game"] = (clean_df["minute_out"] + 1 - clean_df["minute_in"]).astype("int64")
clean_df["minute_in_window"] = pd.Categorical(
    minute_in_window,
    categories=minute_window_order,
    ordered=True,
)
clean_df["minute_out_window"] = pd.Categorical(
    minute_out_window,
    categories=minute_window_order,
    ordered=True,
)

clean_df = clean_df.drop(
    columns=[
        "checkpoint_period",
        "checkpoint_min",
        "minute_in",
        "minute_out",
    ]
)

tables["players_quarters_final_step1.csv"] = clean_df

replacement_columns = [
    "minutes_in_game",
    "minute_in_window",
    "minute_out_window",
]

replacement_missingness = pd.DataFrame(
    {
        "column": replacement_columns,
        "missing_count": [int(clean_df[column].isna().sum()) for column in replacement_columns],
        "n_unique": [int(clean_df[column].nunique(dropna=True)) for column in replacement_columns],
    }
)

display(Markdown("### Step-1 Modeling Table"))
print(f"shape = {clean_df.shape}")
display(replacement_missingness)
preview_columns = [
    "player_appearance_id",
    "checkpoint",
    "minutes_in_game",
    "minute_in_H1_15",
    "minute_in_H2_30",
    "minute_out_H2_45",
    "minute_out_ET2_15",
    "minute_in_window",
    "minute_out_window",
]

display(clean_df[preview_columns].head(10))


### Step-1 Modeling Table

shape = (3486, 48)


,column,missing_count,n_unique
0,minutes_in_game,0,78
1,minute_in_window,0,6
2,minute_out_window,0,7


,player_appearance_id,checkpoint,minutes_in_game,minute_in_H1_15,minute_in_H2_30,minute_out_H2_45,minute_out_ET2_15,minute_in_window,minute_out_window
0,39421,H1_15,90,1,0,1,0,H1_15,H2_45
1,39422,H1_15,90,1,0,1,0,H1_15,H2_45
2,39423,H1_15,90,1,0,1,0,H1_15,H2_45
3,39424,H1_15,90,1,0,1,0,H1_15,H2_45
4,39425,H1_15,75,1,0,0,0,H1_15,H2_30
5,39426,H1_15,87,1,0,1,0,H1_15,H2_45
6,39429,H1_15,90,1,0,1,0,H1_15,H2_45
7,39431,H1_15,80,1,0,1,0,H1_15,H2_45
8,39433,H1_15,87,1,0,1,0,H1_15,H2_45
9,39435,H1_15,90,1,0,1,0,H1_15,H2_45


In [142]:
minute_indicator_columns_to_drop = [
    "minute_in_H1_15",
    "minute_in_H1_30",
    "minute_in_H1_45",
    "minute_in_H2_15",
    "minute_in_H2_30",
    "minute_in_H2_45",
    "minute_in_ET1_15",
    "minute_in_ET2_15",
    "minute_out_H1_15",
    "minute_out_H1_30",
    "minute_out_H1_45",
    "minute_out_H2_15",
    "minute_out_H2_30",
    "minute_out_H2_45",
    "minute_out_ET1_15",
    "minute_out_ET2_15",
]

clean_df = clean_df.drop(columns=minute_indicator_columns_to_drop)
tables["players_quarters_final_step1.csv"] = clean_df

display(Markdown("### Step-1 Modeling Table After Dropping Minute Indicators"))
print(f"shape = {clean_df.shape}")
display(clean_df[["player_appearance_id", "checkpoint", "minutes_in_game", "minute_in_window", "minute_out_window"]].head(10))


### Step-1 Modeling Table After Dropping Minute Indicators

shape = (3486, 32)


,player_appearance_id,checkpoint,minutes_in_game,minute_in_window,minute_out_window
0,39421,H1_15,90,H1_15,H2_45
1,39422,H1_15,90,H1_15,H2_45
2,39423,H1_15,90,H1_15,H2_45
3,39424,H1_15,90,H1_15,H2_45
4,39425,H1_15,75,H1_15,H2_30
5,39426,H1_15,87,H1_15,H2_45
6,39429,H1_15,90,H1_15,H2_45
7,39431,H1_15,80,H1_15,H2_45
8,39433,H1_15,87,H1_15,H2_45
9,39435,H1_15,90,H1_15,H2_45


In [143]:
pass_df = tables["player_appearance_pass.csv"].copy()

period_offsets = {
    "half_1": 0,
    "half_2": 45,
    "extra_time_1": 90,
    "extra_time_2": 105,
}

pass_df["minute_abs"] = pass_df["minute"] + pass_df["period"].map(period_offsets).astype("int64")


def map_absolute_minute_to_checkpoint(minute_abs: int) -> str | None:
    for checkpoint_label, (start_minute, end_minute) in checkpoint_windows.items():
        if start_minute <= minute_abs <= end_minute:
            return checkpoint_label
    return None


pass_df["checkpoint"] = pass_df["minute_abs"].map(map_absolute_minute_to_checkpoint)
pass_df = pass_df[pass_df["checkpoint"].isin(df["checkpoint"].astype(str).unique())].copy()

passed_counts = (
    pass_df.groupby(["player_appearance_id", "checkpoint"], dropna=False)
    .agg(
        last15_pass_passed=("id", "size"),
        last15_pass_passed_accurate=("accurate", "sum"),
    )
    .reset_index()
)

received_counts = (
    pass_df.dropna(subset=["addressee_player_appearance_id"])
    .assign(addressee_player_appearance_id=lambda x: x["addressee_player_appearance_id"].astype("int64"))
    .groupby(["addressee_player_appearance_id", "checkpoint"], dropna=False)
    .agg(
        last15_pass_received=("id", "size"),
        last15_pass_received_accurate=("accurate", "sum"),
    )
    .reset_index()
    .rename(columns={"addressee_player_appearance_id": "player_appearance_id"})
)

clean_df = clean_df.merge(
    passed_counts,
    on=["player_appearance_id", "checkpoint"],
    how="left",
)
clean_df = clean_df.merge(
    received_counts,
    on=["player_appearance_id", "checkpoint"],
    how="left",
)

pass_feature_columns = [
    "last15_pass_received",
    "last15_pass_passed",
    "last15_pass_received_accurate",
    "last15_pass_passed_accurate",
]

for column in pass_feature_columns:
    clean_df[column] = clean_df[column].fillna(0).astype("int64")

tables["players_quarters_final_step1.csv"] = clean_df

display(Markdown("### Step-1 Modeling Table With Pass Features"))
print(f"shape = {clean_df.shape}")
display(
    clean_df[
        [
            "player_appearance_id",
            "checkpoint",
            "minutes_in_game",
            *pass_feature_columns,
        ]
    ].head(10)
)
display(clean_df[pass_feature_columns].describe().T)


### Step-1 Modeling Table With Pass Features

shape = (3486, 36)


,player_appearance_id,checkpoint,minutes_in_game,last15_pass_received,last15_pass_passed,last15_pass_received_accurate,last15_pass_passed_accurate
0,39421,H1_15,90,4,2,2,2
1,39422,H1_15,90,7,7,4,5
2,39423,H1_15,90,7,7,6,7
3,39424,H1_15,90,4,4,3,3
4,39425,H1_15,75,1,0,1,0
5,39426,H1_15,87,5,4,3,3
6,39429,H1_15,90,2,2,2,2
7,39431,H1_15,80,1,0,1,0
8,39433,H1_15,87,5,3,3,1
9,39435,H1_15,90,4,7,3,5


,count,mean,std,min,25%,50%,75%,max
last15_pass_received,3486.0,7.019220,5.029945,0.0,3.0,6.0,9.0,35.0
last15_pass_passed,3486.0,7.037292,5.171413,0.0,3.0,6.0,10.0,36.0
last15_pass_received_accurate,3486.0,5.912794,4.327334,0.0,3.0,5.0,8.0,33.0
last15_pass_passed_accurate,3486.0,5.925129,4.612983,0.0,2.0,5.0,8.0,33.0


In [144]:
checkpoint_order_for_cumsum = [
    checkpoint_label
    for checkpoint_label in minute_window_order
    if checkpoint_label in set(clean_df["checkpoint"].astype(str))
]

cumulative_pass_feature_map = {
    "last15_pass_passed": "cumul_pass_passed",
    "last15_pass_passed_accurate": "cumul_pass_passed_accurate",
    "last15_pass_received": "cumul_pass_received",
    "last15_pass_received_accurate": "cumul_pass_received_accurate",
}

clean_df_with_order = clean_df.assign(
    _row_order=range(len(clean_df)),
    _checkpoint_order=pd.Categorical(
        clean_df["checkpoint"].astype(str),
        categories=checkpoint_order_for_cumsum,
        ordered=True,
    ),
).sort_values(["player_appearance_id", "_checkpoint_order", "_row_order"])

cumulative_values = clean_df_with_order[list(cumulative_pass_feature_map.keys())].groupby(
    clean_df_with_order["player_appearance_id"]
).cumsum()
cumulative_values = cumulative_values.rename(columns=cumulative_pass_feature_map)

clean_df_with_order = pd.concat(
    [clean_df_with_order, cumulative_values],
    axis=1,
).sort_values("_row_order")

clean_df = clean_df_with_order.drop(columns=["_row_order", "_checkpoint_order"])
tables["players_quarters_final_step1.csv"] = clean_df

cumulative_pass_columns = list(cumulative_pass_feature_map.values())

display(Markdown("### Step-1 Modeling Table With Cumulative Pass Features"))
print(f"shape = {clean_df.shape}")
display(
    clean_df[
        [
            "player_appearance_id",
            "checkpoint",
            "last15_pass_passed",
            "cumul_pass_passed",
            "last15_pass_received",
            "cumul_pass_received",
            "last15_pass_passed_accurate",
            "cumul_pass_passed_accurate",
            "last15_pass_received_accurate",
            "cumul_pass_received_accurate",
        ]
    ].head(12)
)
display(clean_df[cumulative_pass_columns].describe().T)


### Step-1 Modeling Table With Cumulative Pass Features

shape = (3486, 40)


,player_appearance_id,checkpoint,last15_pass_passed,cumul_pass_passed,last15_pass_received,cumul_pass_received,last15_pass_passed_accurate,cumul_pass_passed_accurate,last15_pass_received_accurate,cumul_pass_received_accurate
0,39421,H1_15,2,2,4,4,2,2,2,2
1,39422,H1_15,7,7,7,7,5,5,4,4
2,39423,H1_15,7,7,7,7,7,7,6,6
3,39424,H1_15,4,4,4,4,3,3,3,3
4,39425,H1_15,0,0,1,1,0,0,1,1
5,39426,H1_15,4,4,5,5,3,3,3,3
6,39429,H1_15,2,2,2,2,2,2,2,2
7,39431,H1_15,0,0,1,1,0,0,1,1
8,39433,H1_15,3,3,5,5,1,1,3,3
9,39435,H1_15,7,7,4,4,5,5,3,3


,count,mean,std,min,25%,50%,75%,max
cumul_pass_passed,3486.0,20.813827,17.617930,0.0,8.0,16.0,29.0,111.0
cumul_pass_passed_accurate,3486.0,17.605565,15.472423,0.0,6.0,13.0,25.0,91.0
cumul_pass_received,3486.0,20.676420,17.219646,0.0,8.0,16.0,28.0,114.0
cumul_pass_received_accurate,3486.0,17.433448,14.395660,0.0,7.0,13.0,24.0,89.0


In [145]:
export_path = DATA_DIR / "players_quarters_final_step1.csv"
clean_df.to_csv(export_path, index=False)
print(f"Saved clean_df to: {export_path}")
print(f"shape = {clean_df.shape}")


Saved clean_df to: /Users/jan/Documents/competitions/hackatons/WEC_26/data/players_quarters_final_step1.csv
shape = (3486, 40)
